1.2 EDA(Exploratory Data Analysis)

1.2.1 데이터 구조 파악

데이터셋 컬럼 이해, 변수 유형 확인, 데이터셋간 관계 파악

담당자: 김찬진, 성원준, 여승현, 이원일

일정: 6 / 24 - 6 / 25

In [1]:
# 최소 패키지들
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import datetime

In [2]:
# 1. 현재로부터 5년 계산
end = datetime.date.today()
start = end - datetime.timedelta(days=5*365+1)

In [3]:
# 2-1. yfinance.download 방식으로 5년치 apple 주가 데이터 불러오고 기술통계 확인
df_aapl_download = yf.download('AAPL', start=start, end=end)
print(df_aapl_download.info())
print(df_aapl_download.describe())
df_aapl_download.head()

/tmp/ipykernel_1239/3807240666.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df_aapl_download = yf.download('AAPL', start=start, end=end)
[*********************100%***********************]  1 of 1 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1255 entries, 2020-06-25 to 2025-06-24
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   (Close, AAPL)   1255 non-null   float64
 1   (High, AAPL)    1255 non-null   float64
 2   (Low, AAPL)     1255 non-null   float64
 3   (Open, AAPL)    1255 non-null   float64
 4   (Volume, AAPL)  1255 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 58.8 KB
None
Price         Close         High          Low         Open        Volume
Ticker         AAPL         AAPL         AAPL         AAPL          AAPL
count   1255.000000  1255.000000  1255.000000  1255.000000  1.255000e+03
mean     165.708147   167.408842   163.828350   165.557093  7.954750e+07
std       37.338773    37.517210    37.063514    37.258106  4.084625e+07
min       85.938210    88.013588    85.367126    85.845870  2.323470e+07
25%      138.837654   140.801945   136.383751   138.986115  5.20763

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2020-06-25,88.662407,88.701291,86.895674,87.656319,137522400
2020-06-26,85.938210,88.779082,85.789965,88.557936,205256800
2020-06-29,87.918808,88.013588,85.367126,85.845870,130646000
2020-06-30,88.652687,88.939453,87.486207,87.505645,140223200
2020-07-01,88.485016,89.274822,88.436417,88.730466,110737200


In [4]:
# 2-2. yfinance.Ticker.history()로 5년치 불러오고 기술통계
aapl = yf.Ticker('AAPL')
df_aapl_history = aapl.history(period='5y', interval='1d')
print(df_aapl_history.info())
print(df_aapl_history.describe())
df_aapl_history.head()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1255 entries, 2020-06-25 00:00:00-04:00 to 2025-06-24 00:00:00-04:00
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1255 non-null   float64
 1   High          1255 non-null   float64
 2   Low           1255 non-null   float64
 3   Close         1255 non-null   float64
 4   Volume        1255 non-null   int64  
 5   Dividends     1255 non-null   float64
 6   Stock Splits  1255 non-null   float64
dtypes: float64(6), int64(1)
memory usage: 78.4 KB
None
              Open         High          Low        Close        Volume  \
count  1255.000000  1255.000000  1255.000000  1255.000000  1.255000e+03   
mean    165.557093   167.408842   163.828350   165.708147  7.954750e+07   
std      37.258106    37.517210    37.063515    37.338774  4.084625e+07   
min      85.845856    88.013573    85.367111    85.938202  2.323470e+07   
25%     138.986131   140.801

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2020-06-25 00:00:00-04:00,87.656335,88.701306,86.895689,88.662422,137522400,0.0,0.0
2020-06-26 00:00:00-04:00,88.557928,88.779075,85.789958,85.938202,205256800,0.0,0.0
2020-06-29 00:00:00-04:00,85.845856,88.013573,85.367111,87.918793,130646000,0.0,0.0
2020-06-30 00:00:00-04:00,87.505645,88.939453,87.486207,88.652687,140223200,0.0,0.0
2020-07-01 00:00:00-04:00,88.730481,89.274837,88.436432,88.485031,110737200,0.0,0.0


배당금은 대부분이 0에 속한다. 일년에 자주 있는 배당금 지급이 아니다 보니 이것이 앞으로 어떻게 유의하게 쓰거나 처리해야하나를 위해 분석.

In [5]:
# 배당금 컬럼에 대한 빈도 등 시간적 분석
# 배당금 지급된 날 추출
#yf history는 날짜가 인덱스로 들어와 편한 검색을 위해 reset_index를 한다.
dividend_df = df_aapl_history.reset_index() 
dividend_df = dividend_df[dividend_df['Dividends'] > 0].copy()

In [6]:
# 연도 추출
dividend_df['Year'] = dividend_df['Date'].dt.year
# 연도별 배당 지급일 리스트
dividend_df

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Year
30,2020-08-07 00:00:00-04:00,110.241487,110.699184,107.405232,108.203766,198045600,0.205,0.0,2020
94,2020-11-06 00:00:00-05:00,115.421372,116.279811,113.285021,115.782310,114457900,0.205,0.0,2020
155,2021-02-05 00:00:00-05:00,134.185396,134.253775,132.729721,133.608978,75693800,0.205,0.0,2021
218,2021-05-07 00:00:00-04:00,128.052280,128.453502,126.711562,127.425964,78973300,0.220,0.0,2021
281,2021-08-06 00:00:00-04:00,143.435469,144.180329,142.729807,143.229645,54126800,0.220,0.0,2021
345,2021-11-05 00:00:00-04:00,149.082414,149.386681,147.286238,148.483688,65463900,0.220,0.0,2021
407,2022-02-04 00:00:00-05:00,168.721266,171.099573,167.738500,169.419037,82465400,0.220,0.0,2022
470,2022-05-06 00:00:00-04:00,153.546604,156.922452,151.745497,154.796555,116124600,0.230,0.0,2022
532,2022-08-05 00:00:00-04:00,160.856030,163.457953,160.649052,162.965164,56697000,0.230,0.0,2022
596,2022-11-04 00:00:00-04:00,140.272961,140.845546,132.661564,136.610413,140814800,0.230,0.0,2022


In [7]:
dividend_df.describe()

,Open,High,Low,Close,Volume,Dividends,Stock Splits,Year
count,20.000000,20.000000,20.000000,20.000000,2.000000e+01,20.000000,20.0,20.000000
mean,166.782634,168.066734,164.976573,166.661972,7.365480e+07,0.231750,0.0,2022.500000
std,35.104139,35.420919,35.380646,35.338829,4.070059e+07,0.016406,0.0,1.538968
min,110.241487,110.699184,107.405232,108.203766,3.311560e+07,0.205000,0.0,2020.000000
25%,142.644842,143.346633,140.229785,141.574837,4.944408e+07,0.220000,0.0,2021.000000
50%,164.788648,167.278763,164.193776,166.192101,6.061325e+07,0.230000,0.0,2022.500000
75%,184.908854,186.122622,183.364676,185.712749,7.984632e+07,0.242500,0.0,2024.000000
max,229.269351,230.288004,226.902445,227.351852,1.980456e+08,0.260000,0.0,2025.000000


In [8]:
df_aapl_history['Dividends'].value_counts()

Dividends
0.000    1235
0.220       4
0.230       4
0.240       4
0.250       4
0.205       3
0.260       1
Name: count, dtype: int64

매년 4회가 2,5,8,11월 초에 있는것이 확인된다.
이 시기에 등락이 큰지 어떤 처리를 해야하는지를 확인한다.

In [9]:
df_adj = aapl.history(period='5y', interval='1d', auto_adjust=True)     # 보정된 주가
df_raw = aapl.history(period='5y', interval='1d', auto_adjust=False)    # 원본 주가

In [21]:
df_raw

,Open,High,Low,Close,Adj Close,Volume,Dividends,Stock Splits,Close_prev,drop_amount,is_dividend_day,Open_today,drop_by_open,drop_by_close
Date,,,,,,,,,,,,,,
2020-06-25 00:00:00-04:00,90.175003,91.250000,89.392502,91.209999,88.662422,137522400,0.0,0.0,NaN,NaN,False,90.175003,NaN,NaN
2020-06-26 00:00:00-04:00,91.102501,91.330002,88.254997,88.407501,85.938202,205256800,0.0,0.0,91.209999,2.802498,False,91.102501,0.107498,2.802498
2020-06-29 00:00:00-04:00,88.312500,90.542503,87.820000,90.445000,87.918793,130646000,0.0,0.0,88.407501,-2.037498,False,88.312500,0.095001,-2.037498
2020-06-30 00:00:00-04:00,90.019997,91.495003,90.000000,91.199997,88.652687,140223200,0.0,0.0,90.445000,-0.754997,False,90.019997,0.425003,-0.754997
2020-07-01 00:00:00-04:00,91.279999,91.839996,90.977501,91.027496,88.485031,110737200,0.0,0.0,91.199997,0.172501,False,91.279999,-0.080002,0.172501
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-06-17 00:00:00-04:00,197.199997,198.389999,195.210007,195.639999,195.639999,38856200,0.0,0.0,198.419998,2.779999,False,197.199997,1.220001,2.779999
2025-06-18 00:00:00-04:00,195.940002,197.570007,195.070007,196.580002,196.580002,45394700,0.0,0.0,195.639999,-0.940002,False,195.940002,-0.300003,-0.940002
2025-06-20 00:00:00-04:00,198.240005,201.699997,196.860001,201.000000,201.000000,96813500,0.0,0.0,196.580002,-4.419998,False,198.240005,-1.660004,-4.419998


In [22]:
# 배당일 당일과 전일 Close 비교
df_raw['Close_prev'] = df_raw['Close'].shift(1)
df_raw['Open_today'] = df_raw['Open']
df_raw['drop_by_open'] = df_raw['Close_prev'] - df_raw['Open_today']
df_raw['drop_by_close'] = df_raw['Close_prev'] - df_raw['Close']
df_raw['Adj_diff'] = df_raw['Close'] - df_raw['Adj Close']  # 조정된 변화

df_raw['is_dividend_day'] = df_raw['Dividends'] > 0


df_raw[df_raw['is_dividend_day']][[
    'Close_prev', 'Open_today', 'Close', 
    'drop_by_open', 'drop_by_close', 
    'Dividends', 'Adj Close', 'Adj_diff'
]]

,Close_prev,Open_today,Close,drop_by_open,drop_by_close,Dividends,Adj Close,Adj_diff
Date,,,,,,,,
2020-08-07 00:00:00-04:00,113.902496,113.205002,111.112503,0.697495,2.789993,0.205,108.203766,2.908737
2020-11-06 00:00:00-05:00,119.029999,118.320000,118.690002,0.709999,0.339996,0.205,115.782310,2.907692
2021-02-05 00:00:00-05:00,137.389999,137.350006,136.759995,0.039993,0.630005,0.205,133.608978,3.151016
2021-05-07 00:00:00-04:00,129.740005,130.850006,130.210007,-1.110001,-0.470001,0.220,127.425964,2.784042
2021-08-06 00:00:00-04:00,147.059998,146.350006,146.139999,0.709991,0.919998,0.220,143.229645,2.910355
2021-11-05 00:00:00-04:00,150.960007,151.889999,151.279999,-0.929993,-0.319992,0.220,148.483688,2.796310
2022-02-04 00:00:00-05:00,172.899994,171.679993,172.389999,1.220001,0.509995,0.220,169.419037,2.970963
2022-05-06 00:00:00-04:00,156.770004,156.009995,157.279999,0.760010,-0.509995,0.230,154.796555,2.483444
2022-08-05 00:00:00-04:00,165.809998,163.210007,165.350006,2.599991,0.459991,0.230,162.965164,2.384842


> 위 실험은 `auto_adjust=True`가 기본값임을 확인하기 위한 내부 검증이었으며,
> 모델 구성에는 사용되지 않음.

## 📌 yfinance에서 `auto_adjust=True` 설정에 대한 정리

본 분석에서 사용한 `yfinance` API의 `history()` 또는 `download()` 함수는  
**기본적으로 `auto_adjust=True`** 설정이 적용되어 있습니다.

이는 다음과 같은 의미를 갖습니다:

- **배당금 지급(배당락)**, **주식 분할** 등의 이벤트가 주가에 영향을 줄 경우,
- `Close` 컬럼이 이미 그 영향을 **자동으로 보정(adjust)** 해서 제공됩니다.
- 따라서 `Adj Close` 컬럼은 따로 존재하지 않으며, `Close`가 그 역할을 대신합니다.

해당 내용은 아래 Medium 글에 자세히 설명되어 있습니다:  
👉 [Why Adj Close Disappeared in yfinance, and How to Adapt](https://medium.com/@josue.monte/why-adj-close-disappeared-in-yfinance-and-how-to-adapt-6baebf1939f6)

### ✅ 결론
`yfinance`에서 별도 설정 없이 `Close`를 사용하는 것으로  
배당 및 분할 조정이 완료된 시계열을 그대로 사용할 수 있으므로,  
`auto_adjust=True` 설정은 **기본값 그대로 사용**하면 됩니다.


📌 현재 프로젝트에서는 `Dividends` 컬럼을 예측 변수로 사용하지 않습니다.

- 배당 지급은 1년에 4회 이하로 희소하게 발생하며,
- 실제 주가 변동도 배당금과 일치하지 않아 예측에 유의미한 신호가 되지 않습니다.
- 따라서 모델 입력에서는 제거하고,
  필요 시 감성 분석/이벤트 해석용으로만 참고할 예정입니다.

| 컬럼명              | 설명                  | 의미 / 해석                    | 예측에 사용?              |
| ---------------- | ------------------- | -------------------------- | -------------------- |
| **Open**         | 시가 (해당 거래일 첫 거래 가격) | 당일 장 시작 시 투자자 기대 반영        | ✅                    |
| **High**         | 고가                  | 해당 거래일 중 최고 가격             | ✅                    |
| **Low**          | 저가                  | 해당 거래일 중 최저 가격             | ✅                    |
| **Close**        | 종가 (장 마감가)          | 투자 심리 총합이 반영된 최종가, 가장 중요   | ✅ (예측 타겟)            |
| **Volume**       | 거래량                 | 해당 거래일 총 거래 주식 수           | ✅시장 활기 지표, **이상치 주의** |
| **Dividends**    | 배당금                 | 해당 날짜 지급된 배당금              | 🔸대부분 날짜에 0 → 이벤트 분석용  |
| **Stock Splits** | 액면분할                | 주식 분할 이벤트 (예: 4.0은 4:1 분할) | 🔸급변동 원인 → 시각화/주석용     |